# AnimationStudio — Pre-flight validation (one clear image per step)

Run this notebook **before** the full generation notebook. It walks each stage
of the stack and produces a single clear test image so you can confirm the
pipeline works end to end — without burning hours on a full Phase-1 run.

Steps:

1. GPU / VRAM available + studio installed
2. ComfyUI installed
3. Flux model downloaded to the Colab disk
4. ComfyUI server started and fully answering the API
5. The model is visible to ComfyUI (right filename, right loader)
6. The workflow is **Flux-correct** (cfg=1.0, scheduler=simple — SD-style
   CFG values are why Flux output comes out blurred/over-saturated)
7. **One** 1024×1024 test image is generated, saved to the repo, and shown
   inline
8. Automatic sharpness check on that image

**If you see `Connection refused` at any step:** the Colab VM was recycled and
killed the server subprocess. Just re-run the failing cell — steps 4–7 now
auto-restart the server when it is down.

If step 8 shows a sharp, visible image → proceed to the main notebook.
If it shows a flat/blurry image → stop; the troubleshooting list at the end
applies.

In [ ]:
#@title 0. Settings

import os
import subprocess
import sys
import time
import shutil
from pathlib import Path

REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}
BRANCH = "colab-gpu"  #@param ["colab-gpu", "master"]

COMFYUI_PORT = 8188  #@param {type:"integer"}
TEST_SIZE = 1024  #@param {type:"integer"}
TEST_SEED = 42  #@param {type:"integer"}
TEST_PROMPT = "Lily Bunny, cute anthropomorphic white rabbit child, fluffy fur, big round expressive eyes, soft studio lighting, crisp clean 3D render, bright cheerful colors, high detail, sharp focus"  #@param {type:"string"}
TEST_NEGATIVE = "blurry, out of focus, low quality, deformed, distorted, text, watermark, logo"  #@param {type:"string"}

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"
COMFY = f"{WORK}/comfyui"

if REPO_URL.startswith("https://github.com/YOUR_ORG/"):
    raise SystemExit("Set REPO_URL in Cell 0 before running.")


def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)

In [ ]:
#@title 1. STEP 1 — Clone repo, install studio, check GPU/VRAM

os.chdir(WORK)
if not os.path.isdir(REPO):
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q", "requests", "pillow", "numpy", "IPython"])

import torch
assert torch.cuda.is_available(), "CUDA not available on this runtime"
print("PASS: GPU =", torch.cuda.get_device_name(0))
print("      VRAM =", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
#@title 2. STEP 2 — Install ComfyUI (and ComfyUI-GGUF on master)

if not os.path.isdir(COMFY):
    run(["git", "clone", "--depth", "1",
         "https://github.com/comfyanonymous/ComfyUI.git", COMFY])
run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY}/requirements.txt"])

if BRANCH == "master":
    gguf = f"{COMFY}/custom_nodes/ComfyUI-GGUF"
    if not os.path.isdir(gguf):
        run(["git", "clone", "--depth", "1",
             "https://github.com/city96/ComfyUI-GGUF.git", gguf])
    run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{gguf}/requirements.txt"])
print("PASS: ComfyUI installed at", COMFY)

In [ ]:
#@title 3. STEP 3 — Download the Flux model (Colab disk)

MODELS = {
    "colab-gpu": {
        "checkpoints/flux1-dev.safetensors":
            "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
    },
    "master": {
        "checkpoints/flux1-dev-Q4_K_S.gguf":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q4_K_S.gguf",
        "clip/clip_l.safetensors":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/clip_l.safetensors",
        "clip/t5xxl_fp16.safetensors":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/t5xxl_fp16.safetensors",
        "vae/ae.safetensors":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/ae.safetensors",
    },
}[BRANCH]

for rel, url in MODELS.items():
    cached = f"{WORK}/models/{rel}"
    link = f"{COMFY}/models/{rel}"
    if not (os.path.exists(cached) and os.path.getsize(cached) > 0):
        os.makedirs(os.path.dirname(cached), exist_ok=True)
        print(f"Downloading {rel} ...")
        run(["wget", "-q", "-c", "-O", cached, url])
    os.makedirs(os.path.dirname(link), exist_ok=True)
    if os.path.lexists(link) and not os.path.islink(link):
        os.remove(link)
    if not os.path.islink(link):
        try:
            os.symlink(cached, link)
        except OSError:
            shutil.copyfile(cached, link)
    print(f"OK {rel} ({os.path.getsize(cached) / 1e9:.2f} GB)")

MODEL_FILE = os.path.basename(list(MODELS)[0])
print("PASS: model file =", MODEL_FILE)

In [ ]:
#@title 5. STEP 5 — Verify the model is visible to ComfyUI

ensure_comfyui_up()

info = requests.get(f"http://127.0.0.1:{COMFYUI_PORT}/object_info", timeout=60).json()


def options(node, key):
    try:
        return sorted(list(info[node]["input"]["required"][key][0]))
    except Exception:
        return []


checkpoints = options("CheckpointLoaderSimple", "ckpt_name")
unets = options("UnetLoaderGGUF", "unet_name")
clips = options("DualCLIPLoader", "clip_name1")

print("checkpoints:", checkpoints)
print("gguf unets: ", unets)
print("dual clips: ", clips)

visible = MODEL_FILE in checkpoints or MODEL_FILE in unets
print("PASS: model visible to ComfyUI" if visible else
      "FAIL: model file not found by ComfyUI — check Cell 3 symlinks")
assert visible, f"Expected to see {MODEL_FILE} in the loader lists above"


In [ ]:
#@title 5. STEP 5 — Verify the model is visible to ComfyUI

ensure_comfyui_up()

def _options(node, key):
    try:
        info = requests.get(f"http://127.0.0.1:{COMFYUI_PORT}/object_info", timeout=60).json()
        return sorted(list(info[node]["input"]["required"][key][0]))
    except Exception:
        return []

checkpoints = _options("CheckpointLoaderSimple", "ckpt_name")
unets = _options("UnetLoaderGGUF", "unet_name")
clips = _options("DualCLIPLoader", "clip_name1")

print("checkpoints:", checkpoints)
print("gguf unets: ", unets)
print("dual clips: ", clips)

visible = MODEL_FILE in checkpoints or MODEL_FILE in unets
print("PASS: model visible to ComfyUI" if visible else
      "FAIL: model file not found by ComfyUI — check Cell 3 symlinks")
assert visible, f"Expected to see {MODEL_FILE} in the loader lists above"

In [ ]:
#@title 6. STEP 6 — Build a Flux-correct workflow (cfg=1.0, simple)

from src.generation_engine.base import GenerationInput
from src.generation_engine.comfy_backend import ComfyUIBackend

backend = ComfyUIBackend(server_url=f"http://127.0.0.1:{COMFYUI_PORT}")

gen_input = GenerationInput(
    prompt=TEST_PROMPT,
    negative_prompt=TEST_NEGATIVE,
    seed=TEST_SEED,
    num_images=1,
    width=TEST_SIZE,
    height=TEST_SIZE,
)
workflow = backend._build_workflow(gen_input, asset_type="")

# Point the loader at the model we actually installed (fp8 or GGUF).
for node in workflow.values():
    if node.get("class_type") in ("CheckpointLoaderSimple", "UnetLoaderGGUF"):
        key = ("unet_name" if node["class_type"] == "UnetLoaderGGUF" else "ckpt_name")
        node["inputs"][key] = MODEL_FILE
# Re-run the GGUF rewire now that the filename is the real one.
backend._upgrade_to_gguf_if_needed(workflow)

ks = workflow["3"]["inputs"]
print("KSampler :", {k: ks[k] for k in ("steps", "cfg", "sampler_name", "scheduler", "denoise")})
print("Latent   :", workflow["5"]["class_type"], workflow["5"]["inputs"]["width"], "x", workflow["5"]["inputs"]["height"])
print("Loader   :", [(k, v["class_type"], list(v.get("inputs", {}).values())) for k, v in workflow.items()
                     if v.get("class_type") in ("CheckpointLoaderSimple", "UnetLoaderGGUF", "DualCLIPLoader", "VAELoader")])
assert ks["cfg"] == 1.0 and ks["scheduler"] == "simple", "Workflow is NOT Flux-correct"
print("PASS: Flux-correct workflow ready")

In [ ]:
#@title 7. STEP 7 — Generate ONE clear test image (via ComfyUI)

ensure_comfyui_up()

out = backend.generate(gen_input, asset_type="")
assert out.images, f"No image returned: {out.metadata}"

img = out.images[0]
dst_dir = Path(REPO) / "Universe" / "_validate"
dst_dir.mkdir(parents=True, exist_ok=True)
dst = dst_dir / f"smoke_{TEST_SEED}.png"
img.save(dst, format="PNG")
print("saved:", dst)
print("size :", img.size, "|", f"{dst.stat().st_size / 1024:.1f} KB")

from IPython.display import Image as IPImage, display
display(IPImage(filename=str(dst)))
print("PASS: image generated and saved")

In [ ]:
#@title 8. STEP 8 — Automatic sharpness check (blur detection)

import numpy as np
from PIL import Image as PILImage, ImageFilter

gray = PILImage.open(dst).convert("L")
edges = gray.filter(ImageFilter.FIND_EDGES)
score = float(np.asarray(edges, dtype=np.float32).var())

if score > 60:
    verdict = "SHARP — clear, detailed"
elif score > 15:
    verdict = "CHECK — some detail, decide visually above"
else:
    verdict = "BLURRY / FLAT — do NOT proceed"

print("edge-variance sharpness score:", round(score, 1))
print("verdict:", verdict)
print()
print("Compare: a solid-color placeholder (mock backend) scores ~0-50.")
print("A clear Flux image usually scores hundreds to thousands.")

## Next steps

- **Sharp and clear above?** → Run the main notebook
  (`AnimationStudio_Colab.ipynb`) Cells 9–11 for the full Phase-1 run. The
  workflows it uses now carry the same Flux-correct settings this notebook
  verified.
- **BLURRY / no image?** → check, in order:
  1. `comfyui.log` tail: `!tail -n 40 /content/comfyui.log`
  2. Cell 3: model file size is non-zero and the symlink exists
  3. Cell 5: the model name appears in the loader list
  4. VRAM: 16 GB T4 is tight for fp8 Flux — close other notebooks/VMs
  5. The old `mock:` placeholder PNGs in `Universe/...` are *expected*; newly
     generated files overwrite them via `--persist-images`.